# 02 — Train your first detector

This notebook walks through the SAME steps as `src/detector/train.py`, but one cell at a time so you can see what each does. Once you understand it, you can just run the script with `python -m src.detector.train`.

Run from the `notebooks/` folder, after `01_explore.ipynb`.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from src import config

In [ ]:
# 1) Load + clean — only read the 11 columns we need to avoid OOM on large datasets
needed_cols = set(config.SURICATA_ALIGNED_FEATURES + [config.LABEL_COLUMN])
csv_paths = sorted(config.DATA_DIR.glob('**/*.csv'))

frames = []
for p in csv_paths:
    frame = pd.read_csv(p, usecols=lambda col: col.strip() in needed_cols, low_memory=False)
    frame.columns = [c.strip() for c in frame.columns]
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=config.SURICATA_ALIGNED_FEATURES)
print(df.shape)

In [ ]:
# 2) Features (X) and binary target (y): 0 = benign, 1 = attack
X = df[config.SURICATA_ALIGNED_FEATURES].astype(float)
y = (df[config.LABEL_COLUMN].str.strip() != config.BENIGN_LABEL).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
print('train:', X_train.shape, 'test:', X_test.shape)

In [ ]:
# 3) Train the Random Forest (this is the whole 'AI training' step!)
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print('done')

In [ ]:
# 4) Evaluate honestly. Watch the 'attack' recall and the false-positive count.
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['benign','attack']))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['benign','attack'])

In [ ]:
# 5) Which features matter most? (also useful context for the incident report)
pd.Series(model.feature_importances_, index=config.SURICATA_ALIGNED_FEATURES).sort_values().plot(kind='barh')

In [ ]:
# 6) Turn a probability into the project's 0-100 score, and save the model.
import joblib, json
proba = model.predict_proba(X_test.iloc[:5])[:,1]
print('example scores (0-100):', (proba*100).round().astype(int))

config.MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(model, config.MODEL_PATH)
config.FEATURE_COLUMNS_PATH.write_text(json.dumps(config.SURICATA_ALIGNED_FEATURES, indent=2))
print('saved ->', config.MODEL_PATH)

Now the model is saved, the command-line pipeline can use it:
```
python -m src.detector.suricata_reader --demo
```